# 🌍 ExoHabitAI – ML Model Training Pipeline

**Objective:** Select, train, evaluate, and finalize the best-performing ML model for predicting the habitability potential of exoplanets.

| | |
|---|---|
| **Input** | `data/processed/preprocessed.csv` |
| **Task** | Binary Classification — Habitable (1) vs Non-Habitable (0) |
| **Models** | Logistic Regression · Decision Tree · Random Forest · GradientBoosting (XGBoost-equivalent) · SVM |
| **Outputs** | `.pkl` models · `habitability_ranked.csv` · evaluation plots |

---
### Pipeline
1. Imports & Setup
2. Dataset Preparation — X/y split, train/test 80-20
3. Baseline Models — Logistic Regression, Decision Tree
4. Primary Models — Random Forest, GradientBoosting, SVM
5. Feature Scaling & Pipelines
6. Model Training & Saving
7. Model Evaluation — Accuracy, Precision, Recall, F1, ROC-AUC
8. Mandatory Outputs — Confusion Matrix, Classification Report, ROC Curve
9. Hyperparameter Tuning — GridSearchCV
10. Model Comparison & Selection
11. Habitability Scoring & Ranking
12. Model Interpretability — Feature Importance

---
## 1. Imports & Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm             import SVC
from sklearn.utils           import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Directory setup
os.makedirs('models',          exist_ok=True)
os.makedirs('data/processed',  exist_ok=True)

RANDOM_STATE = 42
print('✓ All imports successful')

---
## 2. Dataset Preparation

**Input file:** `data/processed/preprocessed.csv`  
Steps: separate X and y → handle class imbalance → 80/20 train-test split with `random_state=42`

In [ ]:
# ── Load preprocessed data ────────────────────────────────────────────────
df = pd.read_csv('preprocessed.csv')

print(f'Dataset shape : {df.shape}')
print(f'\nTarget distribution:')
vc = df['target'].value_counts()
print(f'  Non-Habitable (0) : {vc.get(0, 0)}')
print(f'  Habitable     (1) : {vc.get(1, 0)}')
df.head()

In [ ]:
# ── Separate features (X) and target (y) ─────────────────────────────────
X = df.drop(columns=['planet_name', 'target'])
y = df['target']

feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')

In [ ]:
# ── Handle Class Imbalance via Oversampling ───────────────────────────────
# Minority class (habitable) is oversampled to improve model recall
df_combined  = pd.concat([X, y], axis=1)
majority     = df_combined[df_combined['target'] == 0]
minority     = df_combined[df_combined['target'] == 1]

minority_up  = resample(
    minority, replace=True,
    n_samples=len(majority) // 4,   # bring to ~20% ratio
    random_state=RANDOM_STATE
)
df_balanced  = pd.concat([majority, minority_up]).sample(frac=1, random_state=RANDOM_STATE)

X_bal = df_balanced.drop(columns=['target'])
y_bal = df_balanced['target']

print(f'Before balancing : {y.value_counts().to_dict()}')
print(f'After  balancing : {y_bal.value_counts().to_dict()}')

In [ ]:
# ── Train / Test Split — 80% training · 20% testing ──────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y_bal        # preserve class ratio in both splits
)

print(f'Training set : {X_train.shape[0]} samples  →  {y_train.value_counts().to_dict()}')
print(f'Test set     : {X_test.shape[0]}  samples  →  {y_test.value_counts().to_dict()}')

In [ ]:
# ── Class Distribution Plot ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels  = ['Non-Habitable (0)', 'Habitable (1)']
colors  = ['#E74C3C', '#2ECC71']

for ax, counts, title in zip(
    axes,
    [y.value_counts().sort_index(), y_bal.value_counts().sort_index()],
    ['Before Balancing', 'After Oversampling']
):
    bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(int(bar.get_height())), ha='center', fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_ylabel('Count')

fig.suptitle('Class Distribution — Habitability Labels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Baseline Models

**Purpose:** Establish reference performance before using advanced models.  
Baseline models: **Logistic Regression** and **Decision Tree (shallow, max_depth=4)**

In [ ]:
# ── Evaluation Helper ─────────────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Fit model, return metrics dict + predictions."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    metrics = {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_te, y_pred,    zero_division=0), 4),
        'F1'       : round(f1_score(y_te, y_pred,        zero_division=0), 4),
        'ROC-AUC'  : round(roc_auc_score(y_te, y_prob),  4) if y_prob is not None else 'N/A',
    }
    return metrics, model, y_pred, y_prob

all_results    = []
trained_models = {}
print('✓ evaluate_model() helper ready')

In [ ]:
# ── Baseline 1: Logistic Regression ──────────────────────────────────────
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, class_weight='balanced',
                                  random_state=RANDOM_STATE))
])

m, trained_models['Logistic Regression'], _, _ = evaluate_model(
    'Logistic Regression', lr_pipeline, X_train, y_train, X_test, y_test
)
all_results.append(m)

print('Logistic Regression Baseline:')
for k, v in m.items():
    if k != 'Model': print(f'  {k:<12}: {v}')

In [ ]:
# ── Baseline 2: Decision Tree (shallow, max_depth=4) ─────────────────────
dt_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    DecisionTreeClassifier(max_depth=4, class_weight='balanced',
                                       random_state=RANDOM_STATE))
])

m, trained_models['Decision Tree'], _, _ = evaluate_model(
    'Decision Tree', dt_pipeline, X_train, y_train, X_test, y_test
)
all_results.append(m)

print('Decision Tree Baseline:')
for k, v in m.items():
    if k != 'Model': print(f'  {k:<12}: {v}')

print('\nBaseline benchmark set ✓ — now comparing advanced models against these scores.')

---
## 4. Primary Model Candidates

Training **at least two mandatory** primary models:  
**4.1 Random Forest** — handles non-linearity, robust to noise, provides feature importance  
**4.2 GradientBoosting** — XGBoost-equivalent in scikit-learn, high performance on structured tabular data  
**4.3 SVM** — optional, strong on smaller high-dimensional datasets  

All wrapped in **scikit-learn Pipelines** to prevent data leakage.

In [ ]:
# ── 4.1 Random Forest Classifier ─────────────────────────────────────────
# Handles non-linearity well · Robust to noise · Provides feature importance
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(
        n_estimators  = 100,
        max_depth     = 10,
        class_weight  = 'balanced',
        random_state  = RANDOM_STATE
    ))
])

m, trained_models['Random Forest'], _, _ = evaluate_model(
    'Random Forest', rf_pipeline, X_train, y_train, X_test, y_test
)
all_results.append(m)

print('Random Forest:')
for k, v in m.items():
    if k != 'Model': print(f'  {k:<12}: {v}')

In [ ]:
# ── 4.2 GradientBoosting Classifier (XGBoost-equivalent) ─────────────────
# High performance on structured data · Handles complex feature interactions
# Note: sklearn's GradientBoostingClassifier mirrors XGBoost's boosting mechanism
gb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(
        n_estimators  = 100,
        learning_rate = 0.1,
        max_depth     = 4,
        random_state  = RANDOM_STATE
    ))
])

m, trained_models['GradientBoosting'], _, _ = evaluate_model(
    'GradientBoosting', gb_pipeline, X_train, y_train, X_test, y_test
)
all_results.append(m)

print('GradientBoosting (XGBoost-equivalent):')
for k, v in m.items():
    if k != 'Model': print(f'  {k:<12}: {v}')

In [ ]:
# ── 4.3 Support Vector Machine (Optional) ────────────────────────────────
# Strong on smaller high-dimensional datasets · uses MinMaxScaler
svm_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('clf',    SVC(
        kernel        = 'rbf',
        class_weight  = 'balanced',
        probability   = True,
        random_state  = RANDOM_STATE
    ))
])

m, trained_models['SVM'], _, _ = evaluate_model(
    'SVM', svm_pipeline, X_train, y_train, X_test, y_test
)
all_results.append(m)

print('SVM (Optional):')
for k, v in m.items():
    if k != 'Model': print(f'  {k:<12}: {v}')

---
## 5. Model Evaluation

Evaluating all models on: **Accuracy · Precision · Recall · F1-Score · ROC-AUC**

Mandatory outputs: **Confusion Matrix · Classification Report · ROC Curve**

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results).set_index('Model')
results_df = results_df.sort_values('F1', ascending=False)

print('\n' + '='*65)
print('      BASELINE vs PRIMARY MODEL COMPARISON')
print('='*65)
print(results_df.to_string())
print('='*65)
results_df

In [ ]:
# ── Metric Comparison Bar Chart ───────────────────────────────────────────
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
plot_df     = results_df[metric_cols].astype(float)

fig, ax = plt.subplots(figsize=(14, 5))
x       = np.arange(len(plot_df))
width   = 0.15
palette = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

for i, col in enumerate(metric_cols):
    bars = ax.bar(x + i * width, plot_df[col], width,
                  label=col, color=palette[i], alpha=0.9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(plot_df.index, fontsize=10, rotation=10, ha='right')
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score')
ax.set_title('Model Performance — All Metrics', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='0.8 threshold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices (Mandatory Output) ─────────────────────────────────
n_models    = len(trained_models)
ncols       = 3
nrows       = (n_models + ncols - 1) // ncols
fig, axes   = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows))
axes        = np.array(axes).flatten()
class_names = ['Non-Habitable', 'Habitable']

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=12, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves (Mandatory Output) ─────────────────────────────────────────
fig, ax  = plt.subplots(figsize=(9, 7))
colors   = ['#3498DB', '#E67E22', '#2ECC71', '#9B59B6', '#E74C3C']

for (name, model), color in zip(trained_models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc     = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.04, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — ExoHabitAI Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Reports (Mandatory Output) ─────────────────────────────
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    print(f'\n{"─"*58}')
    print(f'  Classification Report — {name}')
    print(f'{"─"*58}')
    print(classification_report(
        y_test, y_pred,
        target_names=['Non-Habitable', 'Habitable'],
        zero_division=0
    ))

---
## 6. Model Training & Saving

Saving all trained models using `joblib` into `models/` directory.

In [ ]:
# Save all baseline + primary models
save_map = {
    'logistic_regression' : 'Logistic Regression',
    'decision_tree'       : 'Decision Tree',
    'random_forest'       : 'Random Forest',
    'xgboost'             : 'GradientBoosting',   # stored as xgboost.pkl per docx
    'svm'                 : 'SVM',
}

for filename, model_name in save_map.items():
    if model_name in trained_models:
        path = f'models/{filename}.pkl'
        joblib.dump(trained_models[model_name], path)
        print(f'✓ Saved  models/{filename}.pkl')

---
## 7. Hyperparameter Tuning

Using **GridSearchCV** with 5-fold stratified cross-validation.  
Only tuning after baseline evaluation is complete — tuning the top two models: **Random Forest** and **GradientBoosting**.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Tune Random Forest ────────────────────────────────────────────────────
print('Tuning Random Forest...')

rf_param_grid = {
    'clf__n_estimators'     : [100, 200],
    'clf__max_depth'        : [6, 10, None],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf' : [1, 2],
}

rf_tuned = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('clf', RandomForestClassifier(class_weight='balanced',
                                             random_state=RANDOM_STATE))]),
    rf_param_grid,
    cv      = cv_strategy,
    scoring = 'f1',
    n_jobs  = -1,
    verbose = 0
)
rf_tuned.fit(X_train, y_train)

print(f'Best params : {rf_tuned.best_params_}')
print(f'Best CV F1  : {rf_tuned.best_score_:.4f}')

m, trained_models['Random Forest (Tuned)'], _, _ = evaluate_model(
    'Random Forest (Tuned)', rf_tuned, X_train, y_train, X_test, y_test
)
all_results.append(m)
print(f'Test F1     : {m["F1"]}  |  ROC-AUC: {m["ROC-AUC"]}')

In [ ]:
# ── Tune GradientBoosting (XGBoost params) ────────────────────────────────
print('Tuning GradientBoosting...')

gb_param_grid = {
    'clf__n_estimators' : [100, 200],
    'clf__learning_rate': [0.05, 0.1, 0.2],
    'clf__max_depth'    : [3, 4, 5],
}

gb_tuned = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
    gb_param_grid,
    cv      = cv_strategy,
    scoring = 'f1',
    n_jobs  = -1,
    verbose = 0
)
gb_tuned.fit(X_train, y_train)

print(f'Best params : {gb_tuned.best_params_}')
print(f'Best CV F1  : {gb_tuned.best_score_:.4f}')

m, trained_models['GradientBoosting (Tuned)'], _, _ = evaluate_model(
    'GradientBoosting (Tuned)', gb_tuned, X_train, y_train, X_test, y_test
)
all_results.append(m)
print(f'Test F1     : {m["F1"]}  |  ROC-AUC: {m["ROC-AUC"]}')

---
## 8. Model Comparison & Selection

**Selection criteria:** Best **F1-score** and **Recall** · Stable test performance · No overfitting

In [ ]:
# ── Final Leaderboard ─────────────────────────────────────────────────────
final_df = (
    pd.DataFrame(all_results)
    .set_index('Model')
    .sort_values('F1', ascending=False)
)

print('\n' + '='*65)
print('     FINAL MODEL LEADERBOARD — sorted by F1')
print('='*65)
print(final_df.to_string())
print('='*65)

best_name  = final_df.index[0]
best_model = trained_models[best_name]
print(f'\n🏆 Best Model  : {best_name}')
print(f'   F1-Score    : {final_df.loc[best_name, "F1"]}')
print(f'   Recall      : {final_df.loc[best_name, "Recall"]}')
print(f'   ROC-AUC     : {final_df.loc[best_name, "ROC-AUC"]}')
final_df

In [ ]:
# ── Leaderboard Bar Chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#FFD700' if i == 0 else '#AEB6BF' for i in range(len(final_df))]

bars = ax.barh(
    final_df.index[::-1], final_df['F1'][::-1],
    color=bar_colors[::-1], edgecolor='white', linewidth=1.2
)
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('F1-Score', fontsize=12)
ax.set_title('Model Leaderboard — F1 Score (🥇 = Best Model)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.12)
ax.axvline(0.8, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

print(f'\n✏️  Selection Justification:')
print(f'   {best_name} was selected as the best model based on highest F1-score')
print(f'   and ROC-AUC, demonstrating strong balance between precision and recall')
print(f'   — critical for detecting rare habitable planets without excessive false positives.')

In [ ]:
# Save tuned models and best model
joblib.dump(trained_models['Random Forest (Tuned)'],   'models/random_forest_tuned.pkl')
joblib.dump(trained_models['GradientBoosting (Tuned)'],'models/xgboost_tuned.pkl')
joblib.dump(best_model,                                 'models/best_model.pkl')

print('✓ Saved  models/random_forest_tuned.pkl')
print('✓ Saved  models/xgboost_tuned.pkl')
print(f'✓ Saved  models/best_model.pkl  [{best_name}]')

---
## 9. Habitability Scoring & Ranking

Using the best model's predicted **probabilities** as a habitability score.  
Ranking all exoplanets from most to least habitable.  
Output: `data/processed/habitability_ranked.csv`

In [ ]:
# Run best model on entire (original unbalanced) dataset
X_all    = df.drop(columns=['planet_name', 'target'])
names_all = df['planet_name']

hab_probs = best_model.predict_proba(X_all)[:, 1]
hab_class = best_model.predict(X_all)

ranking_df = pd.DataFrame({
    'planet_name'       : names_all.values,
    'habitability_score': np.round(hab_probs, 4),
    'predicted_class'   : hab_class,
    'label'             : ['Habitable' if c == 1 else 'Non-Habitable' for c in hab_class],
})

# Add key physical features for context
context_cols = ['equilibrium_temperature', 'planet_radius',
                'semi_major_axis', 'insolation_flux', 'habitability_index']
ranking_df = pd.concat(
    [ranking_df.reset_index(drop=True),
     df[context_cols].reset_index(drop=True)], axis=1
)

ranking_df = ranking_df.sort_values('habitability_score', ascending=False).reset_index(drop=True)
ranking_df.index += 1
ranking_df.index.name = 'rank'

# Save
ranking_df.to_csv('data/processed/habitability_ranked.csv')
print(f'✓ Saved data/processed/habitability_ranked.csv')
print(f'  Total planets ranked : {len(ranking_df)}')
print(f'  Predicted habitable  : {(ranking_df["predicted_class"]==1).sum()}')
print('\nTop 15 Most Habitable Exoplanets:')
ranking_df.head(15)

In [ ]:
# ── Top-20 Habitability Score Bar Chart ───────────────────────────────────
from matplotlib.patches import Patch

top20      = ranking_df.head(20)
bar_colors = ['#2ECC71' if c == 1 else '#E74C3C' for c in top20['predicted_class']]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(
    top20['planet_name'][::-1], top20['habitability_score'][::-1],
    color=bar_colors[::-1], edgecolor='white', linewidth=0.8
)
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)

ax.set_xlabel('Habitability Score (Predicted Probability)', fontsize=12)
ax.set_title('Top 20 Exoplanets by Habitability Score — ExoHabitAI', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.12)
legend_elements = [
    Patch(facecolor='#2ECC71', label='Predicted Habitable'),
    Patch(facecolor='#E74C3C', label='Predicted Non-Habitable')
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

---
## 10. Model Interpretability — Feature Importance

Mandatory for final submission: feature importance plot + scientific reasoning.

In [ ]:
# ── Extract Feature Importances ───────────────────────────────────────────
def get_feature_importances(model, feat_names):
    """Works for Pipeline + GridSearchCV wrappers."""
    if hasattr(model, 'best_estimator_'):
        clf = model.best_estimator_['clf']
    else:
        clf = model['clf']
    return clf.feature_importances_

try:
    importances  = get_feature_importances(best_model, feature_names)
    feat_imp_df  = pd.DataFrame({
        'Feature'   : feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    # Plot
    fig, ax = plt.subplots(figsize=(11, 7))
    palette = plt.cm.YlOrRd(np.linspace(0.3, 0.95, len(feat_imp_df)))

    bars = ax.barh(
        feat_imp_df['Feature'][::-1],
        feat_imp_df['Importance'][::-1],
        color=palette, edgecolor='white'
    )
    for bar in bars:
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', fontsize=9)

    ax.set_xlabel('Feature Importance (Gini / Gain)', fontsize=12)
    ax.set_title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\nTop 5 Most Important Features:')
    print(feat_imp_df.head(5).to_string(index=False))

except Exception as e:
    print(f'Feature importance not available for this model type: {e}')

In [ ]:
# ── Scientific Reasoning — Top Feature Interpretations ────────────────────
scientific_reasoning = {
    'equilibrium_temperature' : 'Primary habitability driver. Earth-like life requires ~200–400 K for liquid water. Temperatures far outside this range make habitability unlikely.',
    'insolation_flux'         : 'Amount of stellar energy received. Too high → runaway greenhouse (Venus). Too low → frozen surface (Mars). Optimal: 0.2–5 Earth flux.',
    'habitability_index'      : 'Composite engineered score combining temperature, radius, distance, and stellar size. High values directly indicate Earth-like conditions.',
    'planet_radius'           : 'Determines surface gravity and atmospheric retention. Rocky planets (0.5–2.5 R⊕) can hold breathable atmospheres. Gas giants cannot.',
    'semi_major_axis'         : 'Orbital distance from host star. Defines which stellar flux the planet receives and whether liquid water can persist on its surface.',
    'star_temperature'        : 'G/K-type stars (4000–7000 K, similar to Sun) provide stable, long-lived radiation — ideal for complex life to evolve over billions of years.',
    'star_luminosity'         : 'Stellar size correlates with energy output. Sun-like stars (1 R☉) maintain stable habitable zones for billions of years.',
    'orbital_stability'       : 'Stable, near-circular orbits prevent extreme temperature swings. High eccentricity → oscillating hot/cold extremes hostile to life.',
    'planet_mass'             : 'Affects internal geology (plate tectonics, volcanism) which regulate carbon cycle. Too small → no atmosphere. Too large → gas giant.',
    'stellar_compatibility'   : 'Combined star temperature × size score. High compatibility means a Sun-like star providing consistent energy in a stable habitable zone.',
}

print('Scientific Feature Interpretations')
print('─' * 70)
if 'feat_imp_df' in dir():
    for feat in feat_imp_df['Feature'].head(8):
        desc = scientific_reasoning.get(feat, 'No interpretation available.')
        print(f'\n📌 {feat}')
        print(f'   {desc}')
else:
    for feat, desc in scientific_reasoning.items():
        print(f'\n📌 {feat}')
        print(f'   {desc}')

---
## 11. Cross-Validation Stability Check

In [ ]:
print('5-Fold Stratified Cross-Validation — F1 Score on Training Set')
print('─' * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name in ['Random Forest (Tuned)', 'GradientBoosting (Tuned)', 'SVM']:
    if name in trained_models:
        scores = cross_val_score(
            trained_models[name], X_train, y_train, cv=cv, scoring='f1'
        )
        print(f'{name:<35}  F1: {scores.mean():.4f} ± {scores.std():.4f}  |  Folds: {np.round(scores, 3)}')

---
## 12. Final Output Summary

In [ ]:
import glob

print('=' * 65)
print('  ExoHabitAI — ML Training Pipeline Complete!')
print('=' * 65)
print(f'\n  🏆 Best Model  : {best_name}')
print(f'     F1-Score    : {final_df.loc[best_name, "F1"]}')
print(f'     Recall      : {final_df.loc[best_name, "Recall"]}')
print(f'     ROC-AUC     : {final_df.loc[best_name, "ROC-AUC"]}')
print()
print('  Submitted Files:')
print('  ┌─ models/')
for f in sorted(glob.glob('models/*.pkl')):
    size = os.path.getsize(f) // 1024
    print(f'  │   {os.path.basename(f):<35} ({size} KB)')
print('  ├─ data/processed/')
for f in sorted(glob.glob('data/processed/*.csv')):
    size = os.path.getsize(f) // 1024
    print(f'  │   {os.path.basename(f):<35} ({size} KB)')
print('  └─ notebooks/')
print('      ExoHabitAI_ML_Model_Training.ipynb')
print('=' * 65)